# Project 2: Where are New Yorkers most likely to be evicted?

When a tenant loses a case in housing court, a city marshal can carry out an eviction. NYC publishes a record of every eviction that marshals execute. In this project, I'll combine that with population data to answer two questions:

1. **Do community districts with more people simply have more evictions?**
2. **Once we account for population, which neighborhoods have the highest eviction rates?**

## The data

I'm using two datasets from NYC Open Data:

| Dataset | What's in it |
| --- | --- |
| [Evictions](https://data.cityofnewyork.us/City-Government/Evictions/6z8x-wfk4) | One row per eviction executed by a city marshal since 2017 |
| [New York City Population By Community Districts](https://data.cityofnewyork.us/City-Government/New-York-City-Population-By-Community-Districts/xi7c-iiu2) | The population of each of the city's 59 community districts in each Census from 1970 to 2010 |

The evictions dataset also includes things like addresses, apartment numbers, and marshals' names. I don't need those, and I'd rather not publish them, so I downloaded only the four columns I need using this link:

https://data.cityofnewyork.us/resource/6z8x-wfk4.csv?$select=executed_date,residential_commercial_ind,borough,community_board&$limit=500000

and saved it as `evictions.csv`. The population data was downloaded from the Export button on its page.

## Setup

In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# ensure the visualizations render properly across VSCode, Jupyter Book, etc.
# https://plotly.com/python/renderers/
pio.renderers.default = "notebook_connected+plotly_mimetype"

## Part 1: The evictions data

In [2]:
evictions = pd.read_csv("evictions.csv", parse_dates=["executed_date"])
evictions.head()

,executed_date,residential_commercial_ind,borough,community_board
0,2019-04-18,Residential,BRONX,5.0
1,2017-11-22,Residential,QUEENS,4.0
2,2025-10-10,Residential,BROOKLYN,17.0
3,2018-02-28,Residential,BRONX,3.0
4,2017-01-04,Residential,BRONX,4.0


In [3]:
print(f"Number of evictions: {len(evictions):,}")
print(f"Earliest: {evictions['executed_date'].min():%B %d, %Y}")
print(f"Latest:   {evictions['executed_date'].max():%B %d, %Y}")

Number of evictions: 136,544
Earliest: January 03, 2017
Latest:   October 30, 2026


The data includes both homes and businesses. I'm interested in people losing their homes, so I'll keep only the residential evictions.

In [4]:
evictions["residential_commercial_ind"].value_counts()

residential_commercial_ind
Residential    124652
Commercial      11892
Name: count, dtype: int64

In [5]:
residential = evictions[evictions["residential_commercial_ind"] == "Residential"].copy()

### Which year should I look at?

My first idea was to just count all evictions since 2017 for each neighborhood. Before doing that, let's see how the number of evictions changed from year to year.

In [6]:
residential["year"] = residential["executed_date"].dt.year
per_year = residential.groupby("year").size().reset_index(name="evictions")

fig = px.bar(
    per_year,
    x="year",
    y="evictions",
    title="Residential evictions in NYC by year",
    labels={"year": "Year", "evictions": "Residential evictions"},
)
fig.show()

That was a good thing to check! Evictions almost completely stopped in 2020 and 2021. New York put a [moratorium on evictions](https://en.wikipedia.org/wiki/COVID-19_eviction_moratoriums_in_the_United_States) during the COVID-19 pandemic, and it took a while for them to ramp back up afterwards. The most recent year is also only partially complete.

Adding all of these years together would mix very different periods, so instead I'll focus on **the most recent complete year**.

In [7]:
latest_full_year = residential["year"].max() - 1
recent = residential[residential["year"] == latest_full_year]

print(f"Year: {latest_full_year}")
print(f"Residential evictions that year: {len(recent):,}")

Year: 2025
Residential evictions that year: 17,791


### Missing community districts

Some evictions don't have a community district recorded, so they can't be matched to a neighborhood.

In [8]:
missing_district = recent["community_board"].isna()
print(
    f"Evictions with no community district: {missing_district.sum():,} ({missing_district.mean():.1%})"
)

Evictions with no community district: 1,096 (6.2%)


That's a small share, so I'll leave them out of the neighborhood-level analysis.

In [9]:
recent = recent[~missing_district]
evictions_per_cd = (
    recent.groupby(["borough", "community_board"]).size().reset_index(name="evictions")
)
evictions_per_cd

,borough,community_board,evictions
0,BRONX,1.0,365
1,BRONX,2.0,220
2,BRONX,3.0,453
3,BRONX,4.0,719
4,BRONX,5.0,721
5,BRONX,6.0,469
6,BRONX,7.0,716
7,BRONX,8.0,243
8,BRONX,9.0,570
9,BRONX,10.0,205


## Part 2: The population data

In [10]:
# the population numbers contain commas (like "138,557")
pop_by_cd = pd.read_csv(
    "New_York_City_Population_By_Community_Districts_20261111.csv", thousands=","
)
pop_by_cd.head()

,Borough,CD Number,CD Name,1970 Population,1980 Population,1990 Population,2000 Population,2010 Population
0,Bronx,1,"Melrose, Mott Haven, Port Morris",138557,78441,77214,82159,91497
1,Bronx,2,"Hunts Point, Longwood",99493,34399,39443,46824,52246
2,Bronx,3,"Morrisania, Crotona Park East",150636,53635,57162,68574,79762
3,Bronx,4,"Highbridge, Concourse Village",144207,114312,119962,139563,146441
4,Bronx,5,"University Hts., Fordham, Mt. Hope",121807,107995,118435,128313,128200


The most recent Census in this dataset is 2010, so I'll use that.

In [11]:
populations = pop_by_cd[["Borough", "CD Number", "CD Name", "2010 Population"]]
populations = populations.rename(columns={"2010 Population": "population"})

## Part 3: Combining the two

To match the datasets, I need a column they share. Both have the borough and the community district number, but they're written differently:

- Evictions: `BRONX` and `5.0` (a decimal, because some values were missing)
- Population: `Bronx` and `5`

So I'll make them match before merging.

In [12]:
evictions_per_cd["Borough"] = evictions_per_cd["borough"].str.title()
evictions_per_cd["CD Number"] = evictions_per_cd["community_board"].astype(int)

combined = pd.merge(
    populations,
    evictions_per_cd[["Borough", "CD Number", "evictions"]],
    on=["Borough", "CD Number"],
    how="left",
)
combined

,Borough,CD Number,CD Name,population,evictions
0,Bronx,1,"Melrose, Mott Haven, Port Morris",91497,365
1,Bronx,2,"Hunts Point, Longwood",52246,220
2,Bronx,3,"Morrisania, Crotona Park East",79762,453
3,Bronx,4,"Highbridge, Concourse Village",146441,719
4,Bronx,5,"University Hts., Fordham, Mt. Hope",128200,721
5,Bronx,6,"East Tremont, Belmont",83268,469
6,Bronx,7,"Bedford Park, Norwood, Fordham",139286,716
7,Bronx,8,"Riverdale, Kingsbridge, Marble Hill",101731,243
8,Bronx,9,"Soundview, Parkchester",172298,570
9,Bronx,10,"Throgs Nk., Co-op City, Pelham Bay",120392,205


Let's make sure every community district found a match:

In [13]:
print(f"Community districts: {len(combined)}")
print(f"Without any matched evictions: {combined['evictions'].isna().sum()}")

Community districts: 59
Without any matched evictions: 0


Now calculate the eviction rate: the number of evictions for every 1,000 residents.

In [14]:
combined["evictions_per_1k"] = combined["evictions"] / combined["population"] * 1_000
combined.sort_values("evictions_per_1k", ascending=False).head(10)

,Borough,CD Number,CD Name,population,evictions,evictions_per_1k
2,Bronx,3,"Morrisania, Crotona Park East",79762,453,5.679396
5,Bronx,6,"East Tremont, Belmont",83268,469,5.632416
4,Bronx,5,"University Hts., Fordham, Mt. Hope",128200,721,5.624025
6,Bronx,7,"Bedford Park, Norwood, Fordham",139286,716,5.140502
3,Bronx,4,"Highbridge, Concourse Village",146441,719,4.909827
39,Manhattan,10,Central Harlem,115723,556,4.804576
1,Bronx,2,"Hunts Point, Longwood",52246,220,4.210849
27,Brooklyn,16,"Brownsville, Ocean Hill",86468,358,4.140260
55,Queens,14,"The Rockaways, Broad Channel",114978,471,4.096436
0,Bronx,1,"Melrose, Mott Haven, Port Morris",91497,365,3.989202


## Part 4: Do bigger districts have more evictions?

Each dot below is one community district. If evictions just followed population, the dots would fall neatly along the line.

In [15]:
fig = px.scatter(
    combined,
    x="population",
    y="evictions",
    color="Borough",
    hover_name="CD Name",
    trendline="ols",
    trendline_scope="overall",
    title=f"Residential evictions vs. population, by community district ({latest_full_year})",
    labels={
        "population": "Population (2010 Census)",
        "evictions": f"Residential evictions in {latest_full_year}",
    },
)
fig.show()

In [16]:
correlation = combined["population"].corr(combined["evictions"])
print(f"Correlation between population and evictions: {correlation:.2f}")

Correlation between population and evictions: 0.17


There's only a weak relationship: bigger districts don't reliably have more evictions. Dots far _above_ the line have more evictions than their population would suggest, and many of those are in the Bronx. That's why the eviction _rate_ is a better comparison.

## Part 5: Where are eviction rates highest?

In [17]:
TOP_N = 15
highest_rates = combined.sort_values("evictions_per_1k", ascending=False).head(TOP_N)

fig = px.bar(
    highest_rates,
    x="evictions_per_1k",
    y="CD Name",
    color="Borough",
    orientation="h",
    title=f"The {TOP_N} community districts with the highest eviction rates ({latest_full_year})",
    labels={
        "evictions_per_1k": "Residential evictions per 1,000 residents",
        "CD Name": "Community district",
    },
    height=550,
)
fig.update_yaxes(categoryorder="total ascending")
fig.show()

In [18]:
borough_rates = combined.groupby("Borough")[["evictions", "population"]].sum()
borough_rates["evictions_per_1k"] = (
    borough_rates["evictions"] / borough_rates["population"] * 1_000
)
borough_rates.sort_values("evictions_per_1k", ascending=False)

,evictions,population,evictions_per_1k
Borough,,,
Bronx,5466,1380697,3.958870
Brooklyn,4251,2504609,1.697271
Queens,3760,2241021,1.677807
Manhattan,2643,1577385,1.675558
Staten Island,575,467968,1.228716


## Takeaways

- **Population is a surprisingly weak predictor.** Larger community districts don't reliably have more evictions, and some districts have far more evictions than their size would suggest.
- **Eviction rates are highest in the Bronx.** Once we account for population, Bronx community districts dominate the top of the list, and the Bronx has the highest rate of any borough.
- **The pandemic changed everything for a while.** Evictions nearly stopped during the COVID-19 moratorium, which is why I only looked at the most recent full year instead of adding up every year.

## Caveats

- The population numbers are from the **2010 Census**, while the evictions are much more recent. Neighborhoods have grown and changed since then, so the rates are approximate.
- This only counts evictions **carried out by a marshal**. Many tenants leave before that point, or are pushed out informally, so the true number of people losing their homes is higher.
- A small share of evictions had no community district recorded and couldn't be included.